# Silver: qualidade, SCD2 e snapshots

Branch `chore/estrutura-inicial`. Faça Pull e use uma sessão Python nova para carregar os módulos atualizados. Não reexecute 00–05 nem regenere a massa. Não acrescente reinício dentro de Run all.

Status: implementado e testado localmente; integração Databricks pendente de seus outputs. Consulte `docs/homologacao_domingo.md`.


In [0]:
# CMD 1 — Execute somente este CMD, preencha os campos e depois prossiga.
# catalogo: Catálogo Unity Catalog já validado. Valor recomendado: workspace.
dbutils.widgets.text('catalogo','workspace')
# data_referencia: Data BASE do cenário V2 existente; não é hoje. Valor recomendado: 2026-01-15.
dbutils.widgets.text('data_referencia','2026-01-15')
print("Preencha os widgets antes de executar os próximos CMDs. Run all não pausa para preenchimento.")
# modo: incremental ou reprocessamento; ambos recompõem o cenário completo sem apagar snapshots.
dbutils.widgets.dropdown("modo","incremental",["incremental","reprocessamento"])


In [0]:
# CMD 2 — Localizar o pacote atualizado. Execute dentro da pasta Git.
from pathlib import Path
import sys
atual=Path.cwd().resolve()
raiz=next((p for p in [atual,*atual.parents] if (p/'src/medallion').is_dir() and (p/'pyproject.toml').is_file()),None)
if raiz is None: raise RuntimeError("Abra o notebook dentro da pasta Git")
if str(raiz/'src') not in sys.path: sys.path.insert(0,str(raiz/'src'))
from medallion.config import PipelineConfig
config=PipelineConfig(catalogo=dbutils.widgets.get('catalogo').strip(),data_referencia=dbutils.widgets.get('data_referencia').strip(),modo=dbutils.widgets.get('modo'))
spark.conf.set('spark.sql.session.timeZone','UTC')


In [0]:
# CMD 3 — Executar o módulo e conferir o resultado. Preserve erros; não apague tabelas.
from medallion.pipeline.execute import executar
resultado=executar(spark,config,raiz,'silver')
print("SNAPSHOT_SILVER="+resultado['snapshot_id'])


## Coleta

Exporte IPYNB com outputs, confira os resultados e faça commit/push na branch chore/estrutura-inicial. Envie export e SHA antes de avançar. Nunca confunda saída local com validação Databricks. Snapshots antigos e evidências são preservados. Se ocorrer erro, interrompa e preserve o output.
